# Linly-Dubbing Kaggle WebUI (v4.1 - Technical Fixes Applied)

This notebook is optimized for running **Linly-Dubbing** on Kaggle with **Dual T4 GPUs**. Recent updates include fixes for submodule path conflicts and more robust dependency installation.

### 🚀 Key Improvements in v4.1
- **Submodule Isolation**: Fixed `sys.path` order to prevent submodules (like demucs) from shadowing main project files.
- **Robust Dependencies**: Improved bash syntax and error handling in dependency installation.
- **Resilient AI Model Downloads**: Added retry capability for Hugging Face snapshot downloads to handle server timeouts.
- **Python 3.12 Compatibility**: Enhanced system library installation for building audio tools from source.

### 📋 Execution Guide
1. **Step 1**: Clone repository and check GPU availability
2. **Step 2**: Install all dependencies (Uses robust multi-stage install)
3. **Step 3**: Download required AI models
4. **Step 4**: Launch the WebUI

### ⚙️ Kaggle Setup Requirements
- Enable **GPU T4 x2** in Settings → Accelerator
- Enable **Internet** in Settings → Internet

---

In [11]:
# ============================================================================
# Step 0: 环境检测 (Environment Detection)
# ============================================================================

import os
import sys
import platform
import subprocess
import shutil

print("=" * 60)
print("🔍 Kaggle 运行环境检测")
print("=" * 60)

# Python 信息
print("\n📊 Python 环境:")
print(f"   Python 版本: {sys.version.split()[0]}")
print(f"   Python 路径: {sys.executable}")

# CUDA 信息
print("\n🎮 CUDA 环境:")
try:
    import torch
    print(f"   PyTorch 版本: {torch.__version__}")
    print(f"   CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"   CUDA 版本: {torch.version.cuda}")
        try:
            print(f"   cuDNN 版本: {torch.backends.cudnn.version()}")
        except: print("   cuDNN 版本: 获取失败")
except ImportError:
    print("   ⚠️ PyTorch 未安装")

# 磁盘空间
stat = shutil.disk_usage('/kaggle/working')
print(f"\n💾 磁盘可用空间: {stat.free / (1024**3):.1f} GB")

# 预装 Python 包 (Handle torchvision error gracefully)
print("\n📦 预装 Python 包 (关键):")
for pkg in ['torch', 'torchvision', 'numpy', 'pip']:
    try: 
        m = __import__(pkg)
        print(f"   ✅ {pkg}: {getattr(m, '__version__', 'OK')}")
    except Exception as e:
        print(f"   ⚠️ {pkg}: 导入失败 ({type(e).__name__})")

print("\n" + "=" * 60)
print("✅ 环境检测完成!")
print("=" * 60)

🔍 Kaggle 运行环境检测

📊 Python 环境:
   Python 版本: 3.12.12
   Python 路径: /usr/bin/python3

🎮 CUDA 环境:
   PyTorch 版本: 2.8.0+cu126
   CUDA 可用: True
   CUDA 版本: 12.6
   cuDNN 版本: 91002

💾 磁盘可用空间: 3.3 GB

📦 预装 Python 包 (关键):
   ✅ torch: 2.8.0+cu126
   ✅ torchvision: 0.23.0+cu126
   ✅ numpy: 2.0.2
   ✅ pip: 25.3

✅ 环境检测完成!


In [12]:
# ============================================================================
# Step 1: Clone Repository and Check GPU
# ============================================================================

import os
import torch
import shutil

print("=" * 60)
print("🔍 GPU Detection")
print("=" * 60)
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"✅ Found {gpu_count} GPU(s):")
    for i in range(gpu_count):
        print(f"   - GPU {i}: {torch.cuda.get_device_name(i)}")
else:
    print("❌ No GPU detected! Please enable GPU T4 x2 in settings.")

print("\n" + "=" * 60)
print("📦 Cloning Repository")
print("=" * 60)

# Always remove old directory and clone fresh to ensure latest changes
project_path = '/kaggle/working/Linly-Dubbing'
if os.path.exists(project_path):
    print(f"Removing existing directory: {project_path}")
    shutil.rmtree(project_path)

!cd /kaggle/working && git clone https://github.com/infinite-gaming-studio/Linly-Dubbing.git --depth 1

%cd /kaggle/working/Linly-Dubbing

print("\nInitializing submodules...")
!git submodule update --init --recursive

print("\n✅ Step 1 Complete!")

🔍 GPU Detection
✅ Found 2 GPU(s):
   - GPU 0: Tesla T4
   - GPU 1: Tesla T4

📦 Cloning Repository
Removing existing directory: /kaggle/working/Linly-Dubbing
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into 'Linly-Dubbing'...
remote: Enumerating objects: 1043, done.
remote: Counting objects: 100% (1043/1043), done.
remote: Compressing objects: 100% (863/863), done.
remote: Total 1043 (delta 126), reused 991 (delta 120), pack-reused 0 (from 0)
Receiving objects: 100% (1043/1043), 39.78 MiB | 35.52 MiB/s, done.
Resolving deltas: 100% (126/126), done.
/kaggle/working/Linly-Dubbing

Initializing submodules...
Submodule 'CosyVoice' (https://github.com/FunAudioLLM/CosyVoice.git) registered for path 'CosyVoice'
Cloning into '/kaggle/working/Linly-Dubbing/CosyVoice'...
Submodule path 'CosyVoice': checked ou

In [13]:
# ============================================================================
# Step 2: Install Dependencies (Robust Version for Kaggle)
# ============================================================================

print("=" * 60)
print("📦 Installing System Dependencies")
print("=" * 60)

# 1. Install system tools required for build
!apt-get update -qq
!apt-get install -y -qq build-essential libfst-dev libfst-tools python3-dev ffmpeg \
    libavdevice-dev libavfilter-dev libavformat-dev libavcodec-dev \
    libswresample-dev libswscale-dev libavutil-dev

print("\n🐍 Installing Python Dependencies")
print("=" * 60)

# 2. Update pip tools
!pip install --upgrade -q pip setuptools wheel

# 3. Applying patches to requirements.txt
!sed -i 's/numpy==1.26.3/numpy<2.0.0/g' requirements.txt
!sed -i '/pynini/d' requirements.txt

# 4. Patch TTS for Python 3.12 compatibility
if os.path.exists('submodules/TTS/setup.py'):
    !sed -i 's/Version(python_version) >= Version("3.12"): /Version(python_version) >= Version("3.13"): /g' submodules/TTS/setup.py
    !sed -i 's/python_requires=">=3.9.0, <3.12",/python_requires=">=3.9.0, <3.13",/g' submodules/TTS/setup.py

# 5. Robust pynini installation
print("Installing pynini...")
!if ! pip install -q pynini==2.1.5 --no-cache-dir; then echo "Retry default pynini..."; pip install -q pynini --no-cache-dir; fi

# 6. Install core requirements
print("Installing core requirements...")
!pip install -q -r requirements.txt

# 7. Install submodules with better handling
print("Installing submodules...")
!pip install -q pyannote.audio==3.1.1 faster-whisper==1.0.0
!for sm in submodules/demucs submodules/whisper submodules/whisperX submodules/TTS; do \
    if [ -d "$sm" ]; then \
        echo "   - $sm..."; \
        pip install -q -e "$sm" --no-deps || echo "   ⚠️ Editable install failed for $sm"; \
    fi; \
done

# 8. Final tools
!pip install -q loguru yt-dlp gradio==4.44.1

print("\n✅ Step 2 Complete!")

📦 Installing System Dependencies
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)

🐍 Installing Python Dependencies
Installing pynini...
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'pynini' when getting requirements to build wheel
Retry default pynini...
Installing core requirements...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not

In [14]:
# ============================================================================
# Step 3: Download AI Models
# ============================================================================

print("=" * 60)
print("🤖 Downloading AI Models")
print("=" * 60)

!mkdir -p models/ASR/whisper
wav2vec_path = 'models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth'
if not os.path.exists(wav2vec_path):
    !wget -nc https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth -O {wav2vec_path}

# Use download script
!python scripts/huggingface_download.py

print("\n✅ Step 3 Complete!")

🤖 Downloading AI Models
--2026-01-30 03:09:02--  https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth
Resolving download.pytorch.org (download.pytorch.org)... 18.160.143.101, 18.160.143.21, 18.160.143.107, ...
Connecting to download.pytorch.org (download.pytorch.org)|18.160.143.101|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 377664473 (360M) [application/x-www-form-urlencoded]
Saving to: ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’

models/ASR/whisper/ 100%[===================>] 360.17M   266MB/s    in 1.4s    

2026-01-30 03:09:03 (266 MB/s) - ‘models/ASR/whisper/wav2vec2_fairseq_base_ls960_asr_ls960.pth’ saved [377664473/377664473]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only n

In [15]:
# ============================================================================
# Step 4: Launch WebUI
# ============================================================================

import os
import sys

print("=" * 60)
print("🚀 Launching Linly-Dubbing WebUI")
print("=" * 60)

# Ensure submodules are in path (Fallback for installation issues)
project_root = '/kaggle/working/Linly-Dubbing'
submodule_paths = [
    os.path.join(project_root, 'submodules/demucs'),
    os.path.join(project_root, 'submodules/whisper'),
    os.path.join(project_root, 'submodules/whisperX'),
    os.path.join(project_root, 'submodules/TTS')
]
for p in submodule_paths:
    if os.path.exists(p) and p not in sys.path:
        sys.path.append(p)  # Use append to avoid shadowing project tools

# Set environment variables
os.environ['PYTHONPATH'] = ':'.join(submodule_paths) + (':' + os.environ.get('PYTHONPATH', '') if os.environ.get('PYTHONPATH') else '')
os.environ['MPLBACKEND'] = 'Agg'

if not os.path.exists('.env'):
    !cp env.example .env

print("🌐 Starting Gradio WebUI...")
!python webui.py

🚀 Launching Linly-Dubbing WebUI
🌐 Starting Gradio WebUI...
Traceback (most recent call last):
  File "/kaggle/working/Linly-Dubbing/webui.py", line 17, in <module>
    from tools.step010_demucs_vr import separate_all_audio_under_folder
  File "/kaggle/working/Linly-Dubbing/tools/step010_demucs_vr.py", line 2, in <module>
    from demucs.api import Separator
  File "/kaggle/working/Linly-Dubbing/submodules/demucs/demucs/api.py", line 28, in <module>
    from dora.log import fatal
ModuleNotFoundError: No module named 'dora'
